In [ ]:
# =============================================================================
# STEP 1 - IMPORTS
# =============================================================================

import sqlite3
import pandas as pd

from langchain_ollama import ChatOllama

# =============================================================================
# STEP 2 - DATABASE
# =============================================================================

conn = sqlite3.connect("oil.db")

cursor = conn.cursor()

cursor.execute("""
DROP TABLE IF EXISTS well_production
""")

cursor.execute("""
CREATE TABLE well_production (
    well_name TEXT,
    field_name TEXT,
    production_date TEXT,
    oil_bbl REAL,
    gas_mscf REAL,
    water_bbl REAL,
    hours_on REAL
)
""")

rows = [

    ("WELL-A1","FIELD-X","2026-06-01",1200,800,300,24),
    ("WELL-A2","FIELD-X","2026-06-01",900,600,500,24),
    ("WELL-B1","FIELD-Y","2026-06-01",1500,1100,200,24),

    ("WELL-A1","FIELD-X","2026-06-02",1250,820,320,24),
    ("WELL-A2","FIELD-X","2026-06-02",920,620,510,24),
    ("WELL-B1","FIELD-Y","2026-06-02",1520,1120,210,24),

    ("WELL-A1","FIELD-X","2026-06-03",1230,810,310,24),
    ("WELL-A2","FIELD-X","2026-06-03",910,610,505,24),
    ("WELL-B1","FIELD-Y","2026-06-03",1550,1150,220,24)

]

cursor.executemany("""
INSERT INTO well_production
VALUES (?,?,?,?,?,?,?)
""", rows)

conn.commit()

print("Banco criado")

# =============================================================================
# STEP 3 - SCHEMA
# =============================================================================

schema_df = pd.read_sql(
    "PRAGMA table_info(well_production)",
    conn
)

schema_text = "\n".join(
    schema_df["name"].tolist()
)

print("\nSCHEMA\n")
print(schema_text)

# =============================================================================
# STEP 4 - LLM
# =============================================================================

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)

# =============================================================================
# STEP 5 - QUESTION
# =============================================================================

question = """
Qual poço teve maior produção acumulada de óleo?
"""

# =============================================================================
# STEP 6 - TEXT TO SQL
# =============================================================================

prompt = f"""
Você é especialista em SQL.

Tabela:

well_production

Colunas:

{schema_text}

IMPORTANTE:

Quando a pergunta mencionar:

"produção acumulada"

você DEVE usar:

SUM(oil_bbl)

Exemplo correto:

SELECT
    well_name,
    SUM(oil_bbl) AS total_oil
FROM well_production
GROUP BY well_name
ORDER BY total_oil DESC
LIMIT 1

Retorne APENAS SQL.

Pergunta:

{question}
"""

response = llm.invoke(prompt)

sql = response.content.strip()

print("\n")
print("=" * 80)
print("SQL GERADO")
print("=" * 80)

print(sql)

# =============================================================================
# STEP 7 - EXECUTION
# =============================================================================

print("\n")
print("=" * 80)
print("EXECUTANDO SQL")
print("=" * 80)

try:

    result_df = pd.read_sql(
        sql,
        conn
    )

    print(result_df)

except Exception as e:

    print("\nERRO SQL\n")
    print(e)

# =============================================================================
# STEP 8 - CLOSE
# =============================================================================

conn.close()

Banco criado

SCHEMA

well_name
field_name
production_date
oil_bbl
gas_mscf
water_bbl
hours_on


SQL GERADO
SELECT 
    well_name,
    SUM(oil_bbl) AS total_oil
FROM 
    well_production
GROUP BY 
    well_name
ORDER BY 
    total_oil DESC
LIMIT 1


EXECUTANDO SQL
  well_name  total_oil
0   WELL-B1     4570.0
